# CSCI 2020 — Module 5
## Lecture Notebook — Week 10 (Videos 3–4): CSV/JSON + XLSX Outputs

Run this notebook top-to-bottom.


In [ ]:
# SETUP (do not edit)
from pathlib import Path

ROOT = Path.cwd()
MODULE_DIR = ROOT
DATA_DIR = MODULE_DIR / 'data'
OUTPUT_DIR = MODULE_DIR / 'output'
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print('Working directory:', ROOT)
print('Data dir:', DATA_DIR)
print('Output dir:', OUTPUT_DIR)


## Learning goals
- Understand CSV vs JSON (tables vs nested data)
- Read and write CSV with Python's `csv` module
- Read and write JSON with Python's `json` module
- Open and write `.xlsx` files using `openpyxl`
- Practice the read → transform → write workflow


## CSV vs JSON
- **CSV**: table data (rows/columns), often with a header row
- **JSON**: nested data (lists + dictionaries), common for configs and APIs
> Image placeholder: CSV vs JSON slide.


In [ ]:
# Create a sample CSV in data/ (so demos always work)
import csv
from pathlib import Path

csv_path = DATA_DIR / 'grades.csv'
if not csv_path.exists():
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['name', 'score'])
        w.writerow(['Ada', '90'])
        w.writerow(['Linus', '78'])
        w.writerow(['Grace', '88'])
print('CSV sample:', csv_path)


## CSV reading (standard library)
- Values are read as strings initially
- Convert types when needed
- Use `newline=''` when writing on Windows
> Image placeholder: csv.reader / DictReader slide.


In [ ]:
# Read CSV rows
import csv

rows = []
with open(DATA_DIR / 'grades.csv', 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

print(rows)
print('Types:', type(rows[0]['score']))


## CSV transform + write output
We'll compute a curve (add 5 points, max 100) and write a new CSV.
> Image placeholder: csv.writer / DictWriter output slide.


In [ ]:
import csv

out_csv = OUTPUT_DIR / 'grades_curved.csv'
with open(DATA_DIR / 'grades.csv', 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    data = list(reader)

for r in data:
    score = int(r['score'])
    r['curved_score'] = str(min(100, score + 5))

with open(out_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['name', 'score', 'curved_score'])
    writer.writeheader()
    writer.writerows(data)

print('Wrote:', out_csv)
print(out_csv.read_text(encoding='utf-8'))


## JSON reading/writing
- `json.load` reads from a file handle
- `json.dump` writes to a file handle
- Use `indent=2` for human-readable output
> Image placeholder: json.load / json.dump slide.


In [ ]:
# Create a sample JSON file
import json

json_path = DATA_DIR / 'config.json'
if not json_path.exists():
    sample = {
        'course': 'CSCI2020',
        'module': 5,
        'options': {'verbose': True, 'limit': 10},
    }
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(sample, f, indent=2)
print('JSON sample:', json_path)
print(json_path.read_text(encoding='utf-8'))


In [ ]:
# Read JSON and write a modified JSON output
import json

with open(DATA_DIR / 'config.json', 'r', encoding='utf-8') as f:
    cfg = json.load(f)

cfg['options']['limit'] = 25
cfg['notes'] = 'Edited by Module 5 notebook'

out_json = OUTPUT_DIR / 'config_modified.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, indent=2)

print('Wrote:', out_json)
print(out_json.read_text(encoding='utf-8'))


## XLSX basics (openpyxl)
- XLSX is an Excel workbook format
- `openpyxl` can open, read cells, iterate rows, and write outputs
> Image placeholders: load_workbook, iter_rows, write workbook slides.


In [ ]:
# Create a simple XLSX in data/ if not present
from openpyxl import Workbook, load_workbook

xlsx_path = DATA_DIR / 'scores.xlsx'
if not xlsx_path.exists():
    wb = Workbook()
    ws = wb.active
    ws.title = 'Scores'
    ws.append(['name', 'score'])
    ws.append(['Ada', 90])
    ws.append(['Linus', 78])
    ws.append(['Grace', 88])
    wb.save(xlsx_path)
print('XLSX sample:', xlsx_path)


In [ ]:
# Open workbook and read a few cells
from openpyxl import load_workbook

wb = load_workbook(DATA_DIR / 'scores.xlsx')
ws = wb['Scores']
print('A1:', ws['A1'].value)
print('B2:', ws['B2'].value)


## Iterate rows + write output workbook
We'll compute the average score and write a new workbook in `output/`.


In [ ]:
from openpyxl import Workbook, load_workbook

wb_in = load_workbook(DATA_DIR / 'scores.xlsx')
ws_in = wb_in['Scores']

scores = []
for row in ws_in.iter_rows(min_row=2, values_only=True):
    name, score = row
    scores.append(score)

avg = sum(scores) / len(scores)
print('Average:', avg)

wb_out = Workbook()
ws_out = wb_out.active
ws_out.title = 'Summary'
ws_out.append(['metric', 'value'])
ws_out.append(['average_score', avg])
ws_out.append(['num_students', len(scores)])

out_xlsx = OUTPUT_DIR / 'scores_summary.xlsx'
wb_out.save(out_xlsx)
print('Wrote:', out_xlsx)


## Common gotchas
- CSV: values come in as strings → convert types
- CSV on Windows: use `newline=''` when writing
- JSON: know if you loaded a list vs dict
- XLSX: Excel file may need to be closed to save
- Always save outputs to a new filename


## Wrap-up
- TXT: raw lines; CSV: tables; JSON: structured; XLSX: spreadsheets
- Most workflows: read → transform → write outputs
- Keep `data/` and `output/` folders organized
